In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(device = 0))
print(torch.cuda.device_count())
print(torch.backends.cudnn.version())

In [1]:
# from desktop w/ gpu
import torch
import numpy as np
import cv2
import time
import os
from enum import Enum
from multipledispatch import dispatch
import mrs3 as mr
import interpolation as inter
import utils

%load_ext autoreload
%autoreload 2

lenna_path = 'Lenna_(test_image).png'

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(device = 0))
print(torch.cuda.device_count())
print(torch.backends.cudnn.version())



True
NVIDIA GeForce GTX 1050 Ti
1
8907


In [1]:
# from laptop w/o gpu
import numpy as np
import cv2
import time
import os
from enum import Enum
from PIL import Image
from utils import *
import mrs3 as mr
import interface
from ultralytics import YOLO
import utils

print(cv2.__version__)

%load_ext autoreload
%autoreload 2

4.11.0


In [2]:
cv2.setNumThreads(8)

interface test

In [7]:
utils.unpack_files('2-output-compress/1920x1080.pkg', 'temp-folder')

In [4]:
interface.compress_mult_img_server('1-input-compress-manual', '2-output-compress', manual=True)

before: 440858
after: 70435
compression rate: 0.15976799785872095
before: 219548
after: 135017
compression rate: 0.6149771348406726


In [5]:
interface.compress_mult_img_server('1-input-compress-auto', '2-output-compress', manual=False)


0: 384x640 5 faces, 403.1ms
Speed: 3.6ms preprocess, 403.1ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)
before: 864398
after: 312691
compression rate: 0.3617442428140741

0: 640x640 3 faces, 717.0ms
Speed: 3.0ms preprocess, 717.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)
before: 1575669
after: 272465
compression rate: 0.1729202008797533


In [6]:
interface.restore_imgs_in_folder_server('2-output-compress', '4-output-restore', mrs3_mode=mr.EDSR)

gpu is not available
55.01133728027344 sec taken
복원 이미지 저장 완료: 4-output-restore/1920x1080.png
gpu is not available
18.651713371276855 sec taken
복원 이미지 저장 완료: 4-output-restore/1280x500.png
gpu is not available
7.211017370223999 sec taken
복원 이미지 저장 완료: 4-output-restore/500x500.png
gpu is not available
26.013943195343018 sec taken
복원 이미지 저장 완료: 4-output-restore/960x960.png


In [ ]:
mr.compress_img_mult_tgs_server('1-input-compress-manual/500x500.png', '2-output-compress', scaler=4, pkg_filename='output-imp.pkg')

'2-output-compress/output-imp.pkg'

In [7]:
mr.compress_img_pkg_imgpresso('1-input-compress-manual/500x500.png', '2-output-compress', scaler=4, pkg_filename='output-imp.pkg')

'2-output-compress/output-imp.pkg'

In [ ]:
interface.restore_imgs_in_folder_server('2-output-compress', '4-output-restore', mrs3_mode=mr.EDSR)

gpu is not available
6.939849853515625 sec taken
복원 이미지 저장 완료: 4-output-restore/output-imp.png


face recognition test

In [8]:


image_path = 'sample-images-png/1920x1080.png'
img = cv2.imread(image_path)

results = interface.model(img)

for box in results[0].boxes.xyxy:
    x1, y1, x2, y2 = map(int, box)
    print(f'{x1}, {y1}, {x2}, {y2}')
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)  # 초록색 사각형 그리기

cv2.imshow('Detected Faces', img)
cv2.waitKey(0)
cv2.destroyAllWindows()


0: 384x640 5 faces, 521.7ms
Speed: 11.5ms preprocess, 521.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
385, 201, 499, 391
1292, 228, 1409, 400
445, 91, 573, 261
1088, 316, 1196, 479
905, 329, 1009, 467


In [3]:

import face_recognition
import cv2

# 얼굴 인식할 이미지 파일 경로 설정
image_path = '1-input-compress-auto/1920x1080.png'  # 예: 'your_image.jpg'로 변경

# 이미지 로드 (BGR → RGB 자동 변환)
image = face_recognition.load_image_file(image_path)

# 얼굴 위치 검출 (딥러닝 기반 검출)
face_locations = face_recognition.face_locations(image)

# OpenCV용 이미지로 변환(BGR)
image_cv = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

# 검출된 얼굴 위치에 사각형 그리기
for (top, right, bottom, left) in face_locations:
    cv2.rectangle(image_cv, (left, top), (right, bottom), (0, 255, 0), 3)

# 얼굴 검출 결과 이미지 보기
cv2.imshow('Face Detection', image_cv)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 얼굴 좌표 출력(선택적)
print(face_locations)  # 각각 (top, right, bottom, left) 좌표 리스트


[(254, 1386, 383, 1257), (219, 545, 374, 390), (125, 583, 254, 454)]


In [4]:
import cv2
import numpy as np

# 전역 변수 선언
drawing_points = []  # 현재 누적 클릭 점 리스트
all_points = []      # 모든 그룹의 점 리스트

def mouse_callback(event, x, y, flags, param):
    global drawing_points, all_points
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing_points.append((x, y))
        print(f"Point added: {(x, y)}")
        cv2.circle(img, (x, y), 3, (0, 255, 0), -1)
        cv2.imshow('image', img)
    elif event == cv2.EVENT_RBUTTONDOWN:
        if len(drawing_points) >= 3:
            all_points.append(drawing_points.copy())
            print(f"Target saved: {drawing_points}")
            drawing_points = []
        else:
            print("3개 이상의 점을 먼저 찍어주세요.")

# 테스트용 이미지 로드 (이미지가 없으면 흰 배경 생성)
img = cv2.imread('sample-images-png/500x500.png')
if img is None:
    img = 255 * np.ones((400, 400, 3), dtype=np.uint8)

cv2.namedWindow('image')
cv2.setMouseCallback('image', mouse_callback)
cv2.imshow('image', img)

print("왼쪽 클릭: 점 추가 / 우클릭: 타겟 저장 (3점 이상) / s: 종료&결과 출력")
while True:
    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        if len(drawing_points) >= 3:
            all_points.append(drawing_points.copy())
        print(f"최종 결과: {all_points}")
        break

cv2.destroyAllWindows()


왼쪽 클릭: 점 추가 / 우클릭: 타겟 저장 (3점 이상) / s: 종료&결과 출력
Point added: (287, 215)
Point added: (264, 244)
Point added: (312, 288)
Point added: (386, 255)
Target saved: [(287, 215), (264, 244), (312, 288), (386, 255)]
Point added: (147, 169)
Point added: (119, 234)
Point added: (154, 282)
Point added: (206, 301)
Target saved: [(147, 169), (119, 234), (154, 282), (206, 301)]
Point added: (302, 94)
Point added: (257, 117)
Point added: (312, 175)
Point added: (387, 149)
Target saved: [(302, 94), (257, 117), (312, 175), (387, 149)]
Point added: (221, 387)
Point added: (220, 417)
Point added: (266, 444)
Point added: (319, 404)
Target saved: [(221, 387), (220, 417), (266, 444), (319, 404)]
최종 결과: [[(287, 215), (264, 244), (312, 288), (386, 255)], [(147, 169), (119, 234), (154, 282), (206, 301)], [(302, 94), (257, 117), (312, 175), (387, 149)], [(221, 387), (220, 417), (266, 444), (319, 404)]]


In [ ]:
mr.compress_img_mult_tgs('Downloads/screenshot.png', 'testf', scaler=4, roi_mode=mr.ROI_POLYGON)

In [ ]:
mr.restore_img_mult_tgs('2-output-compress', mr.EDSR, 'testf')

In [ ]:
mr.compress_img_pkg('Downloads/500x500.png', 'testpkg', filename='lenna.pkg', scaler=4, roi_mode=mr.ROI_POLYGON)

In [ ]:
mr.compress_img_pkg_imgpresso('Downloads/500x500.png', 'testpkgimgpresso', filename='lenna.pkg', scaler=4, roi_mode=mr.ROI_POLYGON)

In [ ]:
print(57623/325420)
print(57623/70272)

In [ ]:
mr.compress_img_pkg('Downloads/screenshot.png', 'testpkg', filename='screenshot.pkg', scaler=4, roi_mode=mr.ROI_POLYGON)

In [ ]:
mr.compress_img_mult_tgs('sample-images-png/1920x1080.png', 'testf', scaler=4, roi_mode=mr.ROI_POLYGON)

In [ ]:
mr.restore_img_mult_tgs('testf', mr.EDSR, 'testf')

In [ ]:
# from laptop w/o gpu
import numpy as np
import cv2
import time
import os
from enum import Enum
import matplotlib.pyplot as plt
from collections import defaultdict
from PIL import Image
from utils import *

print(cv2.__version__)

%load_ext autoreload
%autoreload 2

In [ ]:
import cv2
import time



img = cv2.imread('Lenna_(test_image).png')

sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel(f'models/EDSR_x4.pb')

# GPU 가속화를 위한 설정
sr.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
sr.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)

sr.setModel('edsr', 4)

t1 = time.time()
result = sr.upsample(img)

t2 = time.time()
print(t2 - t1)

cv2.imshow('Original Image', img)
cv2.imshow('Super Resolution Image', result)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
cv2.imshow('Original Image', img)
cv2.imshow('Super Resolution Image', result)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
sr = cv2.dnn_superres.DnnSuperResImpl_create()
img = cv2.imread('Lenna_(test_image).png')
img = img[200:400, 200:400]

In [ ]:
cv2.imshow('img', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel('models/EDSR_x4.pb')
sr.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
sr.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)
sr.setModel('edsr', 4)
result = sr.upsample(img)

# cv2.imshow('res', result)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel('models/EDSR_x3.pb')

In [ ]:
def foo():
    return None, None

a, b = foo()
print(a)
print(b)


# TEST

In [ ]:
output_path = 'testfiles'
img_path = 'sample-images-png/1920x1080.png'
scaler = 4

original_part_path = f"{output_path}/original_part.png"
downscaled_part_path = f"{output_path}/downscaled_part.png"

original_part, original_part_loc = select_polygon_roi(img_path)
downscaled_part = downscale_img(img_path, scaler)

cv2.imwrite(original_part_path, original_part)
cv2.imwrite(downscaled_part_path, downscaled_part)

# Original image

In [ ]:
original_img = cv2.imread(img_path)
cv2.imshow('original', original_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
cv2.imshow('downscaled_part', downscaled_part)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
cv2.imshow('original_part', original_part)
cv2.waitKey(0)
cv2.destroyAllWindows()

# File sizes

In [ ]:
original_filesize = os.path.getsize(img_path)
original_part_filesize = os.path.getsize(original_part_path)
downscaled_part_filesize = os.path.getsize(downscaled_part_path)

print(f"original file: {original_filesize}")
print(f"original part file: {original_part_filesize}")

print(f"downscaled part file: {downscaled_part_filesize}")

print(f"compressed ratio: {original_part_filesize / original_filesize} + {downscaled_part_filesize / original_filesize} = {(original_part_filesize + downscaled_part_filesize) / original_filesize}")


# Restored by cv2.resize

In [ ]:
# numpy array[row #, col #] 순

restored_img_by_resize = cv2.imread(downscaled_part_path)
original_h, original_w = restored_img_by_resize.shape[0]*scaler, restored_img_by_resize.shape[1]*scaler
restored_img_by_resize = cv2.resize(restored_img_by_resize, (original_w, original_h), interpolation=cv2.INTER_CUBIC)

cv2.imshow('restored_img_by_resize_only', restored_img_by_resize)
cv2.waitKey(0)
cv2.destroyAllWindows()

# restored_img_by_resize[original_part_loc[], c_from:c_to] = cv2.imread(original_part_path)

cv2.imshow('restored_img_by_resize_mrs3', restored_img_by_resize)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Restored by edsr

In [ ]:
restored_img_by_edsr = upscale_by_edsr(downscaled_part_path, scaler)

cv2.imshow('restored_img_by_edsr_only', restored_img_by_edsr)
cv2.waitKey(0)
cv2.destroyAllWindows()

# restored_img_by_edsr[r_from:r_to, c_from:c_to] = cv2.imread(original_part_path)

r_from, r_to, c_from, c_to = original_part_loc
tmp = restored_img_by_edsr[r_from:r_to, c_from:c_to]
original_part_read = cv2.imread(original_part_path)

tmp = combine_images(original_part_read, tmp)
restored_img_by_edsr[r_from:r_to, c_from:c_to] = tmp


cv2.imshow('restored_img_by_edsr_mrs3', restored_img_by_edsr)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
cv2.imshow('restored_img_by_edsr_mrs3', restored_img_by_edsr)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
# 저해상도 -> 원본 사이즈니까
# 기존 한계가 500x500 이므로
# 2000x2000 을 4배 스케일링하는 것까지 커버가능

# cv2.imwrite('restored_img_by_resize.png', restored_img_by_resize)
cv2.imwrite('restored_img_by_edsr.png', restored_img_by_edsr)

합칠 떄 [0 0 0] 은 스킵하도록 해야함(shape 유지)

In [ ]:
image_path = ''

다른 딥러닝 모델 
- 윤곽/경계
- YOLO
- DEEPSORT
- TRACKING



수정 필요한 거
- 여러개 선택
- 원본 부분 이미지 합칠 때 경계의 부자연스러움 처리(블렌딩, 브러쉬처럼 특정부분 weight 바꾸기)

In [ ]:
img_path = 'Lenna_(test_image).png'


# 경계 블렌딩

우선 아래 방법 적용하지 않은 상태로 구현

- 원본 파트에서 경계에서 떨어진 거리에 따라 블렌딩 비율 다르게
- 경계에 가까울수록 downscaled->upscaled 이미지 비율을 높게 잡고
- 경계에서 멀어질수록 original part 이미지 비율을 높게 설정하여 블렌딩
- 일정 거리 이상으로 멀어지면 100% 원본파트
- 선형, sigmoid 등 함수로 interpolation

In [ ]:
class blend_mode(Enum):
    alpha = 0
    poisson = 1
    laplacian = 2
    grad = 3
    dl = 4

def blending_two_images(src, target, mask, mode = blend_mode.poisson, center = (0, 0)):
    match mode:
        case blend_mode.alpha:
            result = cv2.addWeighted(src, 0.7, target, 0.3, 0, mask=mask)
        case blend_mode.poisson:
            result = cv2.seamlessClone(
                src, target, mask, center, cv2.MIXED_CLONE
            )
        case blend_mode.laplacian:
            # laplacian dist
            return
        case blend_mode.grad:
            # gradient
            return
        case blend_mode.dl:
            # dl model
            return

    return result

In [ ]:

# from keras_cv.models import (
#     ResNet18, ResNet50, YOLOV8Detector, StableDiffusion,
#     DeepLabV3Plus, RetinaNet, ImageClassifier
# )


# lr_image = cv2.imread('Lenna_(test_image).png')
# model = EDSR(scale_factor=4)  # ×4 초해상화
# hr_image = model.predict(lr_image)


In [ ]:
# Hugging Face 모델 다운로드
import keras
# model = keras.models.load_model("https://huggingface.co/keras-io/EDSR")


In [ ]:
from keras_cv.models import YOLOV8Detector
from keras.applications.resnet50 import ResNet50
# model = YOLOV8Detector()
# print(model.device)  # 모델이 할당된 장치 출력


In [ ]:
# Python 인터프리터 또는 스크립트 내에서
import os
print(os.environ.get('LD_LIBRARY_PATH'))


In [ ]:
import keras_cv.src.utils
import keras_cv.src.models

model = keras_cv.models.YOLOV8Detector.from_preset(
    "yolov8n_pascalvoc", 
    bounding_box_format="xyxy" 
)

In [ ]:
import keras_cv
from tensorflow import keras # Keras import 필요
import numpy as np
from keras_cv.utils import image # 이미지 로드/저장 유틸리티

# 1. 모델 로드 (예: Pascal VOC 데이터셋으로 학습된 yolov8n)
# bounding_box_format은 데이터셋 라벨 형식과 맞춰야 함 (일반적으로 "xyxy")
model = keras_cv.models.YOLOV8Detector.from_preset(
    "yolov8n_pascalvoc", 
    bounding_box_format="xyxy" 
)

# 2. 이미지 로드 및 전처리 (Preset이 전처리 레이어 포함)
image = keras.utils.load_img("Lenna_(test_image).png") 
image = np.array(image)
input_batch = np.expand_dims(image, axis=0) # 배치 차원 추가

# 3. 추론 수행
# predict()는 모델 내부의 전처리 및 후처리(NMS 등)를 포함
y_pred = model.predict(input_batch)

# y_pred 형식: 딕셔너리 {'boxes': ..., 'classes': ..., 'confidence': ...}
# bounding_box_format에 따라 box 좌표 형식이 달라짐

# 4. 결과 시각화 (KerasCV 유틸리티 사용)
keras_cv.visualization.plot_bounding_box_gallery(
    input_batch,
    value_range=(0, 255),
    rows=1,
    cols=1,
    y_pred=y_pred,
    scale=5,
    font_scale=0.7,
    bounding_box_format="xyxy",
    class_mapping=model.presets["yolov8n_pascalvoc"].classes, # Preset에 포함된 클래스 매핑 사용
    show=True
)


In [ ]:
std_model = keras_cv.models.StableDiffusion(img_width=512, img_height=512)

def plot_images(images):
    plt.figure(figsize=(20, 20))
    for i in range(len(images)):
        ax = plt.subplot(1, len(images), i + 1)
        plt.imshow(images[i])
        plt.axis("off")

In [ ]:
std_images = std_model.text_to_image("photograph of an astronaut riding a horse", batch_size=3)
plot_images(std_images)

# "photograph of an astronaut riding a horse"
# "photograph of a pig riding a chicken"

In [ ]:
import keras_cv

# EfficientNetV2 백본 로드
# backbone = keras_cv.models.EfficientNetV2Backbone.from_preset("efficientnetv2_b0_imagenet")

# YOLOv8 객체 탐지 모델 로드
# detector = keras_cv.models.YOLOV8Detector.from_preset("yolov8n_pascalvoc", bounding_box_format="xyxy")

# Stable Diffusion 이미지 생성 모델 로드
# sd = keras_cv.models.StableDiffusion(img_width=512, img_height=512)

# https://github.com/keras-team/keras-cv
# 여기에 있는 모델 목록이 기본 제공 모델



In [ ]:
from tensorflow import keras

# Hugging Face 모델 다운로드
model = keras.models.load_model("https://huggingface.co/keras-io/EDSR")


In [ ]:
# del std_model
import gc
gc.collect()

In [ ]:
from keras import backend as K

K.clear_session()

In [ ]:
import mrs3 as mr

In [ ]:
mr.compress_img(img_path='Lenna_(test_image).png', output_path='mrmrmrout', scaler=4, roi_mode=mr.ROI_POLYGON, interpolation=mr.INTER_AREA)

In [ ]:
mr.restore_img(input_path='mrmrmrout', mrs3_mode=mr.EDSR, output_path='')

In [ ]:
"""
3019/232733

401/69500

428/34384

758/125725

1716/127655

585/85001
"""
585/85001

In [ ]:
l = []
len(l)
l.append([])
l[0].append(3324)


In [ ]:
print(l[0])

In [ ]:
contours = []
contour_num = 0

In [ ]:
if len(contours) == contour_num:
    contours.append([])


In [ ]:
contours[contour_num].append('asdf')

In [ ]:
l = [3,23,5,4756,876,5446,765]
print(l[-1])

In [ ]:
if l:
    print('asdf')

r = []
if r:
    print('zxcv')
else:
    print('xcvfzvcx')

In [ ]:
import os
os.path.getsize('models/EDSR_x4.pb') / 1024

In [ ]:
img = cv2.imread('Lenna_(test_image).png')
print(img.itemsize)

In [ ]:
import pynvml
pynvml.nvmlInit()
device_count = pynvml.nvmlDeviceGetCount()

print(device_count)


In [ ]:
for i in range(device_count):
    handle = pynvml.nvmlDeviceGetHandleByIndex(i)
    name = pynvml.nvmlDeviceGetName(handle)
    memory_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    utilization = pynvml.nvmlDeviceGetUtilizationRates(handle)
    
    print(f"GPU {i}: {name.decode('utf-8')}")
    print(f"Memory Total: {memory_info.total / 1024**2:.2f} MB")
    print(f"Memory Used: {memory_info.used / 1024**2:.2f} MB")
    print(f"Memory Free: {memory_info.free / 1024**2:.2f} MB")
    print(f"GPU Utilization: {utilization.gpu}%")
    print(f"Memory Utilization: {utilization.memory}%")
    print()

# NVML 종료
pynvml.nvmlShutdown()

In [ ]:
free_memory = torch.cuda.memory_reserved(0) - torch.cuda.memory_allocated(0)

# print(f"Free GPU Memory: {free_memory / 1024 ** 2:.2f} MB")